
Project : Pentaho Log Intelligence

Layer   : Gold

Notebook: 03_Gold_PentahoLog

Version : 1.0

Description:
Loads raw Pentaho log files from Unity Catalog Volume

into the Gold Delta table.

Author: Ernesto Felipe Garay Cervantes


#### Recibimiento de parametros

In [0]:
import json

dbutils.widgets.text("archivos_nuevos","")

archivos_nuevos = json.loads(dbutils.widgets.get("archivos_nuevos"))

print("====ARCHIVOS RECIBIDOS===")
for archivo in archivos_nuevos:
    print(archivo) 

#### Configuracion

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    current_timestamp
)

In [0]:
CATALOG = "pentaho_logs"

SILVER_TABLE_PENTAHO = "pentaho_logs.silver.silver_logs_pentaho"

GOLD_TABLE_PENTAHO= "pentaho_logs.gold.gold_logs_pentaho"

#### Lectura  de tabla Bronze 

In [0]:
df_pentaho_sl = spark.table(SILVER_TABLE_PENTAHO).filter(col("file_name").isin(archivos_nuevos))

#display(df_pentaho_sl.limit(20))

In [0]:
df_pentaho_sl.printSchema()

#### Creación de identificador de Evento Log Pentaho

In [0]:
from pyspark.sql.functions import (col,lit,sha2,concat_ws,current_timestamp)

df_gold_pentaho = (df_pentaho_sl.withColumn("event_id",sha2(concat_ws("||",col("file_path"),col("hora"),col("Nivel")),256))
                 .withColumn("source_type",lit("PENTAHO_LOG"))
                  .withColumn("gold_timestamp",current_timestamp())
                 )


In [0]:
#display(
#df_gold_pentaho.select(
#          "event_id",
#          "file_name",
#          "application",
#          "fecha",
#          "hora",
#          "Nivel",
#          "proceso_pentaho_log",
#          "descripcion"
 #).limit(20)
#)

In [0]:
df_gold_pentaho = df_gold_pentaho.select( 
      "event_id",
          "file_name",
          "application",
          "fecha",
          "hora",
          "Nivel",
          "proceso_pentaho_log",
          "descripcion"
 )

In [0]:
#display(df_gold_pentaho.limit(20))


#### validación DATAFRAME

In [0]:
# Número de registros
print(f"Total de líneas: {df_gold_pentaho.count():,}")

# Estructura
df_gold_pentaho.printSchema()

#### Creación Tabla GOLD Pentaho_Log

In [0]:
GOLD_TABLE_PENTAHO = "pentaho_logs.gold.gold_logs_pentaho"
(
    df_gold_pentaho.write
        .format("delta")
        .mode("append")
        .saveAsTable(GOLD_TABLE_PENTAHO)
)

In [0]:
#display(spark.table(GOLD_TABLE_PENTAHO).limit(20))